# ScreamingFace · configuration and architecture

See exactly where ScreamingFace stops and its URL4 engine begins: configure one engine, inspect its
registry, distinguish a reusable Fusion recipe from an executed request, and send one real
provider-free URL4 transaction.

## Before you run it

Start the local development stack from the repository root:

```bash
cd packages/screamingface/apps/screamingface-engine
./dev.sh
```

No provider credentials are needed. The only executed expression calls the deterministic
majority-vote route with literal answers. This notebook does not load a benchmark, contact a model,
or use Hugging Face access.

## 1 · Configure one engine

In [ ]:
import json
import os

import httpx
from url4 import Expression, RelExpr, Text, render, src, struct

import screamingface as sf

ENGINE_URL = os.environ.get("SCREAMINGFACE_ENGINE_URL", "http://127.0.0.1:4404")
sf.config(engine=ENGINE_URL)

httpx.get(f"{ENGINE_URL}/healthz", timeout=5).text

`sf.config(...)` stores and validates one HTTP(S) origin. It performs no network
request by itself. Localhost is temporarily the SDK default; the same API selects a future hosted
engine:

```python
sf.config(engine="https://screamingface.example")
```

The explicit health check above is this notebook's first request.

## 2 · The ownership boundary

```text
Researcher process
├─ ScreamingFace SDK
│  ├─ loads and validates engine benchmark manifests
│  ├─ compiles complete benchmark-run URL4
│  ├─ reads provider status and sends connection actions
│  └─ parses and validates the returned report
│
└─ screamingface-engine · persistent Url4Node
   ├─ plaintext URL4 data plane · GET /v1?q=...
   │  ├─ benchmark case data routes
   │  ├─ model routes ── AI Gateway ── model providers
   │  ├─ verified HF tool routes ── Tavily search/extract
   │  ├─ reducer and grader routes
   │  └─ aggregator routes
   └─ JSON connection control plane · /v1/connections/...
      ├─ AI Gateway model-provider credential profiles
      └─ process-local Tavily connection
```

Both planes end at the configured ScreamingFace engine. The SDK never calls providers or AI
Gateway directly. The generic URL4 engine evaluates the graph; the ScreamingFace engine profile
registers the model, tool, reducer, and connection capabilities needed by the SDK.

## 3 · Inspect the engine registry

In [ ]:
registry_response = httpx.get(f"{ENGINE_URL}/.well-known/screamingface", timeout=5)
registry_response.raise_for_status()

{
    "content_type": registry_response.headers["content-type"],
    "raw_plaintext": registry_response.text,
}

In [ ]:
registry = json.loads(registry_response.text)

{
    "schema": registry["schema"],
    "models": sf.models.list(),
    "reducers": registry["reducers"],
    "limits": registry["limits"],
}

The registry is JSON serialized inside a plaintext HTTP body. `sf.models.list()`
fetches and validates the complete `screamingface.registry.v1` document before returning model IDs.

The registry advertises executable model and benchmark routes, response schemas, transport limits,
and provider ownership/auth methods.
Fresh connection status comes from the engine's protected connection API rather than this public
capability document. Loading a benchmark validates its manifest but does not fetch its cases.

## 4 · A Fusion recipe is URL4, but not yet a request

In [ ]:
fusion = sf.Fusion(
    "architecture-example",
    members=["codex/gpt-5.5", "gemini/2.5-flash"],
    reducer=sf.reducers.MajorityVote(),
)

fusion.url4

`fusion.url4` is a canonical parameterized answer recipe. Its `$question` binding is
still unresolved, so displaying it performs no model call. `benchmark.evaluate(fusion, first=...)`
wraps it with the benchmark case route, stable slice, grader, and aggregator.

The returned `report.url4` is the complete shareable run: another compatible engine can read the
same expression and reproduce the selected slice.

## 5 · Build one provider-free URL4 transaction

In [ ]:
expression = render(
    Expression(
        sources=(
            src(
                struct({"member_1": "A", "member_2": "B", "member_3": "A"}),
                name="member_answers",
            ),
            src(
                RelExpr(
                    path="/reducers/majority-vote/1",
                    intent=Text("$member_answers"),
                ),
                name="recipe_answer",
            ),
            src(
                struct(
                    {
                        "schema": "screamingface.recipe-result.v1",
                        "answer": "$recipe_answer",
                    }
                ),
                name="recipe_result",
            ),
        ),
        intent=Text("$recipe_result"),
    )
)

expression

This expression is built from URL4's public Python builders and canonically rendered;
it is not copied from a private ScreamingFace compiler. Its graph binds three literal answers,
passes their resolved object to the registered majority-vote route, and returns one small
structured result.

No model route appears in the graph, so executing it cannot reach AI Gateway, a provider, or
Tavily.

## 6 · Send the encoded GET and inspect the plaintext

In [ ]:
request = httpx.Request("GET", f"{ENGINE_URL}/v1", params={"q": expression})

with httpx.Client(timeout=5) as client:
    response = client.send(request)

response.raise_for_status()
parsed_response = json.loads(response.text)
assert parsed_response == {
    "schema": "screamingface.recipe-result.v1",
    "answer": "A",
}

{
    "url4_expression": expression,
    "encoded_request_url": str(request.url),
    "raw_plaintext": response.text,
    "parsed_result": parsed_response,
}

The complete transactional shape is:

```text
GET /v1?q=<encoded URL4 expression>
```

URL encoding changes only the HTTP representation; the `q` value remains the same URL4 expression.
The engine resolves the graph and returns plaintext. ScreamingFace parses and validates structured
plaintext when it runs a Fusion; this cell performs those two steps visibly for teaching.

## 7 · Responsibility map

| Concern | Owner |
|---|---|
| Benchmark manifest | ScreamingFace SDK from engine registry |
| Benchmark source and references | ScreamingFace engine data route |
| Complete run URL4 compilation | ScreamingFace SDK |
| URL4 graph execution | `screamingface-engine` / URL4 |
| Provider calls | Engine through AI Gateway |
| Provider credential control | SDK through engine to AI Gateway |
| Web research | Route-selected by the engine: OpenRouter managed tools or Tavily on verified HF
routes |
| Grading and aggregation | ScreamingFace engine routes inside URL4 |

Model-backed graders call their judge route inside the same URL4 graph. The SDK never opens a
direct provider or Gateway connection.

## Recap

- `sf.config(...)` selects one URL4 engine.
- The engine registry describes what that deployment can execute.
- `sf.connect()` displays fresh connection state and sends credentials only to that engine.
- model-backed work checks required connections once before spend.
- `fusion.url4` is a reusable parameterized answer recipe.
- `report.url4` is the complete benchmark, slice, Recipe, grading, and aggregation run.
- evaluation sends that expression in one encoded `GET /v1?q=...` transaction.
- successful bodies are plaintext that the SDK parses and validates.
- benchmark data and scoring remain engine-side and reproducible in URL4.

Continue to the quickstart to evaluate GPQA.